# 25일 랜드마크 통합 테이블 (`02_Integrated_Table`)

개강 후 **25일 시점에 재학 중인 학생**을 대상으로, 0~25일까지 관측 가능한 정보로
25일 이후 최종 `Withdrawn` 여부를 예측하기 위한 통합 테이블을 만든다.

결정 근거:
- `work_process/decisions/modeling/0902_01_landmark_n25_final.md`
- `work_process/decisions/target_definition/0829_01_landmark_prediction_target.md`
- `work_process/reports/05_landmark_25_28_32_comparison.md`

데이터 계약(그레인, 타깃 정의, 관측창 제한)은 아래 각 절에서 코드로 직접 구현하고 검증한다.
통합은 SQL 없이 pandas로만 진행한다(2026-09-04 결정: 통합용 SQL은 정본에서 제외).
생성되는 컬럼별 상세 설명(뜻, 결측 조건 등)은 코드에 반복해서 적지 않고
`docs/데이터_사전.md` 8번 섹션에 모아둔다. 컬럼 뜻이 궁금하면 그쪽을 먼저 확인한다.


## 0. 환경설정

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)

CSV_DIR = Path('../../CSV_files')
INTEGRATED_DIR = CSV_DIR / '통합 버전'
LANDMARK_N = 25
KEYS = ['code_module', 'code_presentation', 'id_student']

## 1. 원본 데이터 로드

In [2]:
student_info = pd.read_csv(CSV_DIR / 'studentInfo.csv')
student_registration = pd.read_csv(CSV_DIR / 'studentRegistration.csv')
assessments = pd.read_csv(CSV_DIR / 'assessments.csv')
student_assessment = pd.read_csv(CSV_DIR / 'studentAssessment.csv')
student_vle = pd.read_csv(CSV_DIR / 'studentVle.csv')

for name, df in [
    ('studentInfo', student_info),
    ('studentRegistration', student_registration),
    ('assessments', assessments),
    ('studentAssessment', student_assessment),
    ('studentVle', student_vle),
]:
    print(f'{name:20s} shape={df.shape}')

studentInfo          shape=(32593, 12)
studentRegistration  shape=(32593, 5)
assessments          shape=(206, 6)
studentAssessment    shape=(173912, 5)
studentVle           shape=(10655280, 6)


## 2. 코호트 정의: `eligible_at_25`

기준(`work_process/reports/05_landmark_25_28_32_comparison.md` §2와 동일):

1. `studentInfo` + `studentRegistration`을 `(code_module, code_presentation, id_student)`로 1:1 결합
2. `date_registration > 25`(25일 이후 등록) → 제외
3. `date_unregistration <= 25`(25일까지 이미 이탈) → 제외, 별도 "초기 이탈" 세그먼트로 보관
4. 남은 대상 중 `final_result == 'Withdrawn'`인데 `date_unregistration`이 없는 93건(판정 불명)
   → 주 코호트에서 제외, 별도 "판정 불명" 세그먼트로 보관(민감도 분석용)

기대 결과: 27,661행, 이후 `Withdrawn` 5,247건

In [3]:
base = student_info.merge(
    student_registration, on=KEYS, how='left', validate='one_to_one'
)
assert base.duplicated(KEYS).sum() == 0, '키 중복 발생'

is_late_registration = base['date_registration'] > LANDMARK_N
is_early_leaver = base['date_unregistration'] <= LANDMARK_N

excluded_late_registration_25 = base[is_late_registration].copy()
excluded_early_leaver_25 = base[is_early_leaver].copy()

remaining = base[~(is_late_registration | is_early_leaver)].copy()

is_ambiguous_withdrawn = (
    (remaining['final_result'] == 'Withdrawn')
    & remaining['date_unregistration'].isna()
)
excluded_ambiguous_withdrawn_25 = remaining[is_ambiguous_withdrawn].copy()

eligible_at_25 = remaining[~is_ambiguous_withdrawn].copy()

print('25일 이후 등록 제외      :', len(excluded_late_registration_25))
print('25일까지 이미 이탈 제외  :', len(excluded_early_leaver_25))
print('판정 불명(취소일 없음) 제외:', len(excluded_ambiguous_withdrawn_25))
print('eligible_at_25 행 수      :', len(eligible_at_25))

assert len(eligible_at_25) == 27_661, f'예상 코호트 27,661행과 불일치: {len(eligible_at_25)}'

25일 이후 등록 제외      : 20
25일까지 이미 이탈 제외  : 4819
판정 불명(취소일 없음) 제외: 93
eligible_at_25 행 수      : 27661


## 3. 타깃 정의: `target_churn_after_25`

- 1 = `eligible_at_25` 대상 중 최종 `final_result == 'Withdrawn'` (25일 시점엔 재학 중이었고, 25일 이후 이탈)
- 0 = 최종 `Pass`, `Distinction`, `Fail`

기대 결과: 양성 5,247건

In [4]:
eligible_at_25['target_churn_after_25'] = (
    eligible_at_25['final_result'] == 'Withdrawn'
).astype(int)

print(eligible_at_25['target_churn_after_25'].value_counts())
assert eligible_at_25['target_churn_after_25'].sum() == 5_247, '예상 양성 5,247건과 불일치'

target_churn_after_25
0    22414
1     5247
Name: count, dtype: int64


## 4. VLE 피처 (0~25일)

- `studentVle`에서 `0 <= date <= 25`인 행만 사용 (개강 전 사전 접속은 제외 — 추가 피처 실험에서 효과가
  CV PR-AUC +0.001 수준이라 채택하지 않음, `work_process/reports/07_pipeline_audit_before_final.md` 3.4절)
- 활동량은 행 수가 아니라 `sum_click`의 합으로 계산
컬럼 뜻: `docs/데이터_사전.md` 8번 섹션 "VLE 피처" 표 참고


In [5]:
vle_window = student_vle[
    (student_vle['date'] >= 0) & (student_vle['date'] <= LANDMARK_N)
]

vle_features = (
    vle_window.groupby(KEYS)
    .agg(
        total_click_25=('sum_click', 'sum'),
        active_days_25=('date', 'nunique'),
        distinct_resources_25=('id_site', 'nunique'),
    )
    .reset_index()
)

eligible_at_25 = eligible_at_25.merge(vle_features, on=KEYS, how='left')
for col in ['total_click_25', 'active_days_25', 'distinct_resources_25']:
    eligible_at_25[col] = eligible_at_25[col].fillna(0)

eligible_at_25['no_vle_activity_25'] = (
    eligible_at_25['total_click_25'] == 0
).astype(int)

has_vle = (eligible_at_25['total_click_25'] > 0).sum()
print(f'VLE 활동 보유 학생: {has_vle}명 ({has_vle / len(eligible_at_25):.4%})')

VLE 활동 보유 학생: 26352명 (95.2677%)


## 5. 평가 피처 (0~25일)

- **평가 기회 후보**: `assessments`에서 `assessment_type in (TMA, CMA)`이고 마감일 `date <= 25`인 항목
  (과목-학기 단위). **`weight` 조건은 두지 않는다.**
- **이월(banked) 평가는 학생별로 기회에서 뺀다.** 재수강생이 과거 점수를 이월받은 평가는 이번 학기에
  낼 필요가 없는 평가다. 이월 개수는 `n_banked_25`로 따로 남긴다.
  - `n_opportunity_25` = (과목-학기의 기회 후보 수) − `n_banked_25` → **학생마다 다를 수 있다**
- **실제 제출**(`n_submitted_25`): 기회 후보 중 `is_banked == 0`이고 실제 제출일 `date_submitted <= 25`인
  기록만 사용한다. 마감일이 25일 이내라도 제출이 25일 이후라면 관측 시점엔 아직 미제출이다(시점 누수 방지).
- "평가 기회 없음"과 "실제 미제출"을 구분하기 위해 `n_opportunity_25`, `n_missing_25`,
  `no_assessment_opportunity_25`를 모두 별도 컬럼으로 둔다.

**정의 변경 이력 (2026-09-13, `work_process/decisions/preprocessing/0913_01_assessment_opportunity_banked.md`)**

| 변경 | 이전 | 이후 | 이유 |
|---|---|---|---|
| 가중치 조건 | `weight > 0`만 기회 | `weight` 무관 | BBB 2014J의 첫 TMA(19일)가 이전 학기와 같은 위치인데 가중치만 5 → 0으로 바뀌어, 2014J BBB 전원이 "기회 0"이 되는 구조 단절 발생. 25일 내 가중치 0 평가는 BBB 2014J 1건뿐이라 다른 과목·학기 값은 변하지 않음 |
| 이월 평가 | 기회에는 포함, 제출에서는 제외 | 기회에서도 제외 + `n_banked_25` | 이월받은 재수강생 464명이 "기회는 있었는데 전부 미제출"로 기록돼 실제 미제출자(이탈률 42.2%)와 섞였음(이월자 실제 이탈률 24.8%) |

컬럼 뜻(`avg_score_25` 등): `docs/데이터_사전.md` 8번 섹션 "평가 피처" 표 참고


In [6]:
# 기회 후보: 25일까지 마감된 TMA/CMA (weight 조건 없음 — 5절 정의 변경 이력 참고)
assessment_opportunities = assessments[
    (assessments['assessment_type'].isin(['TMA', 'CMA']))
    & (assessments['date'] <= LANDMARK_N)
].copy()

# 정의 변경의 영향 범위 확인: weight 0 기회 후보가 어느 과목-학기에 있는지
weight0_opportunities = assessment_opportunities[assessment_opportunities['weight'] == 0]
print('weight 0인 기회 후보:')
print(weight0_opportunities[['code_module', 'code_presentation', 'id_assessment', 'assessment_type', 'date']].to_string(index=False))
assert set(zip(weight0_opportunities['code_module'], weight0_opportunities['code_presentation'])) == {('BBB', '2014J')}, \
    'weight 0 기회 후보가 BBB 2014J 외에도 존재 — 정의 변경 영향 범위 재검토 필요'

module_opportunity_counts = (
    assessment_opportunities.groupby(['code_module', 'code_presentation'])
    .size()
    .rename('n_module_opportunity_25')
    .reset_index()
)

opportunity_ids = set(assessment_opportunities['id_assessment'])
opportunity_meta = assessment_opportunities[
    ['id_assessment', 'code_module', 'code_presentation', 'date']
].rename(columns={'date': 'due_date_25'})

# 이월 평가: 학생별로 기회에서 제외할 개수 (이월 기록의 date_submitted는 의미 없는 고정값이라 날짜 조건 없음)
banked_records = student_assessment[
    (student_assessment['is_banked'] == 1)
    & (student_assessment['id_assessment'].isin(opportunity_ids))
].merge(opportunity_meta, on='id_assessment', how='left')
banked_counts = banked_records.groupby(KEYS).size().rename('n_banked_25').reset_index()

eligible_at_25 = eligible_at_25.merge(
    module_opportunity_counts, on=['code_module', 'code_presentation'], how='left'
)
eligible_at_25 = eligible_at_25.merge(banked_counts, on=KEYS, how='left')
eligible_at_25['n_module_opportunity_25'] = eligible_at_25['n_module_opportunity_25'].fillna(0).astype(int)
eligible_at_25['n_banked_25'] = eligible_at_25['n_banked_25'].fillna(0).astype(int)
eligible_at_25['n_opportunity_25'] = (
    eligible_at_25['n_module_opportunity_25'] - eligible_at_25['n_banked_25']
)
assert (eligible_at_25['n_opportunity_25'] >= 0).all(), '이월 수가 과목 기회 수를 초과'
eligible_at_25['no_assessment_opportunity_25'] = (
    eligible_at_25['n_opportunity_25'] == 0
).astype(int)

observed_submissions = student_assessment[
    (student_assessment['is_banked'] == 0)
    & (student_assessment['date_submitted'] <= LANDMARK_N)
    & (student_assessment['id_assessment'].isin(opportunity_ids))
].merge(opportunity_meta, on='id_assessment', how='left')
observed_submissions['submit_delay_25'] = (
    observed_submissions['date_submitted'] - observed_submissions['due_date_25']
)

# 같은 학생이 같은 평가를 이월과 실제 제출로 동시에 갖지 않는지 (이중 계산 방지)
overlap = banked_records[['id_assessment', 'id_student']].merge(
    observed_submissions[['id_assessment', 'id_student']], how='inner'
)
assert overlap.empty, '이월 평가와 실제 제출이 같은 평가에 동시에 존재'

submission_features = (
    observed_submissions.groupby(KEYS)
    .agg(
        n_submitted_25=('id_assessment', 'count'),
        avg_score_25=('score', 'mean'),
        avg_submit_delay_25=('submit_delay_25', 'mean'),
    )
    .reset_index()
)

eligible_at_25 = eligible_at_25.merge(submission_features, on=KEYS, how='left')
eligible_at_25['n_submitted_25'] = (
    eligible_at_25['n_submitted_25'].fillna(0).astype(int)
)
assert (eligible_at_25['n_submitted_25'] <= eligible_at_25['n_opportunity_25']).all(), '제출 수가 기회 수를 초과'
eligible_at_25['n_missing_25'] = (
    eligible_at_25['n_opportunity_25'] - eligible_at_25['n_submitted_25']
)
eligible_at_25['submission_rate_25'] = np.where(
    eligible_at_25['n_opportunity_25'] > 0,
    eligible_at_25['n_submitted_25'] / eligible_at_25['n_opportunity_25'],
    np.nan,
)

has_opportunity = (eligible_at_25['n_opportunity_25'] > 0).sum()
has_submission = (eligible_at_25['n_submitted_25'] > 0).sum()
has_banked = (eligible_at_25['n_banked_25'] > 0).sum()
print()
print(f'유효 평가 기회 보유 학생: {has_opportunity}명 ({has_opportunity / len(eligible_at_25):.4%})')
print(f'실제 제출 이력 보유 학생: {has_submission}명 ({has_submission / len(eligible_at_25):.4%})')
print(f'이월 평가 보유 학생    : {has_banked}명 (이탈률 {eligible_at_25.loc[eligible_at_25["n_banked_25"] > 0, "target_churn_after_25"].mean():.1%})')
print()
print('과목-학기별 평균 기회 수 (이월 반영 전 / 후):')
print(eligible_at_25.groupby(['code_module', 'code_presentation'])[['n_module_opportunity_25', 'n_opportunity_25']].mean().round(3).unstack(0).to_string())
eligible_at_25 = eligible_at_25.drop(columns='n_module_opportunity_25')


weight 0인 기회 후보:
code_module code_presentation  id_assessment assessment_type  date
        BBB             2014J          15020             TMA  19.0

유효 평가 기회 보유 학생: 22208명 (80.2863%)
실제 제출 이력 보유 학생: 19306명 (69.7950%)
이월 평가 보유 학생    : 464명 (이탈률 24.8%)

과목-학기별 평균 기회 수 (이월 반영 전 / 후):
                  n_module_opportunity_25                               n_opportunity_25                                      
code_module                           AAA  BBB  CCC  DDD  EEE  FFF  GGG              AAA    BBB    CCC    DDD  EEE    FFF  GGG
code_presentation                                                                                                             
2013B                                 NaN  1.0  NaN  2.0  NaN  1.0  NaN              NaN  0.995    NaN  1.979  NaN  0.998  NaN
2013J                                 1.0  1.0  NaN  1.0  0.0  1.0  0.0            1.000  0.974    NaN  0.970  0.0  0.975  0.0
2014B                                 NaN  1.0  1.0  1.0  0.0  1.0  0.0         

## 6. 최종 컬럼 정리

`landmark_day`, `eligible_at_25`를 명시 컬럼으로 추가하고, 모델 피처/타깃/감사(audit)용 컬럼을
분리한다. `final_result`, `date_unregistration`은 코호트·타깃을 만드는 재료이므로
**모델 입력 피처로 사용하지 않는다** (`docs/데이터_사전.md` "모델링할 때 절대 피처로 넣으면
안 되는 컬럼" 참고).

In [7]:
eligible_at_25['landmark_day'] = LANDMARK_N
eligible_at_25['eligible_at_25'] = 1

id_cols = ['code_module', 'code_presentation', 'id_student', 'landmark_day']
student_feature_cols = [
    'gender', 'region', 'highest_education', 'imd_band', 'age_band',
    'disability', 'num_of_prev_attempts', 'studied_credits', 'date_registration',
]
vle_feature_cols = [
    'total_click_25', 'active_days_25', 'distinct_resources_25', 'no_vle_activity_25',
]
assessment_feature_cols = [
    'n_opportunity_25', 'n_submitted_25', 'n_missing_25', 'submission_rate_25',
    'avg_score_25', 'avg_submit_delay_25', 'no_assessment_opportunity_25', 'n_banked_25',
]
target_cols = ['eligible_at_25', 'target_churn_after_25']
audit_cols = ['final_result', 'date_unregistration']  # 모델 피처 금지, 추적용으로만 보관

feature_cols = student_feature_cols + vle_feature_cols + assessment_feature_cols
final_cols = id_cols + feature_cols + target_cols + audit_cols

integrated_table_25 = eligible_at_25[final_cols].reset_index(drop=True)
integrated_table_25.head()

,code_module,code_presentation,id_student,landmark_day,gender,region,highest_education,imd_band,age_band,disability,num_of_prev_attempts,studied_credits,date_registration,total_click_25,active_days_25,distinct_resources_25,no_vle_activity_25,n_opportunity_25,n_submitted_25,n_missing_25,submission_rate_25,avg_score_25,avg_submit_delay_25,no_assessment_opportunity_25,n_banked_25,eligible_at_25,target_churn_after_25,final_result,date_unregistration
0,AAA,2013J,11391,25,M,East Anglian Region,HE Qualification,90-100%,55<=,N,0,240,-159.0,303.0,7.0,24.0,0,1,1,0,1.0,78.0,-1.0,0,0,1,0,Pass,NaN
1,AAA,2013J,28400,25,F,Scotland,HE Qualification,20-30%,35-55,N,0,60,-53.0,335.0,11.0,21.0,0,1,1,0,1.0,70.0,3.0,0,0,1,0,Pass,NaN
2,AAA,2013J,31604,25,F,South East Region,A Level or Equivalent,50-60%,35-55,N,0,60,-52.0,309.0,16.0,21.0,0,1,1,0,1.0,72.0,-2.0,0,0,1,0,Pass,NaN
3,AAA,2013J,32885,25,F,West Midlands Region,Lower Than A Level,50-60%,0-35,N,0,60,-176.0,263.0,15.0,17.0,0,1,0,1,0.0,NaN,NaN,0,0,1,0,Pass,NaN
4,AAA,2013J,38053,25,M,Wales,A Level or Equivalent,80-90%,35-55,N,0,60,-110.0,450.0,23.0,20.0,0,1,1,0,1.0,79.0,0.0,0,0,1,0,Pass,NaN


## 7. 검증

- 행 수 27,661 / 양성 5,247
- 키 유일성
- 시점 누수: 모델 피처 컬럼 중 25일 이후 정보가 섞이지 않았는지(설계상 이미 0~25일로 필터링됨을
  재확인하고, 금지 컬럼이 `feature_cols`에 없는지 점검)

In [8]:
assert len(integrated_table_25) == 27_661, '행 수 불일치'
assert integrated_table_25.duplicated(KEYS).sum() == 0, '키 중복 발생'
assert integrated_table_25['target_churn_after_25'].sum() == 5_247, '양성 건수 불일치'

leakage_forbidden = {'final_result', 'date_unregistration', 'target_churn_after_25', 'eligible_at_25'}
leaked = leakage_forbidden & set(feature_cols)
assert not leaked, f'금지 컬럼이 피처에 포함됨: {leaked}'

print('행 수              :', len(integrated_table_25))
print('키 중복             :', integrated_table_25.duplicated(KEYS).sum())
print('양성(target=1) 건수  :', integrated_table_25['target_churn_after_25'].sum())
print('모델 피처 컬럼 수     :', len(feature_cols))
print('검증 통과')

integrated_table_25.isna().sum()[integrated_table_25.isna().sum() > 0]

행 수              : 27661
키 중복             : 0
양성(target=1) 건수  : 5247
모델 피처 컬럼 수     : 21
검증 통과


imd_band                1030
date_registration          7
submission_rate_25      5453
avg_score_25            8362
avg_submit_delay_25     8355
date_unregistration    22413
dtype: int64

## 8. 전체 코호트 단일 파일 저장

정본은 `CSV_files/통합 버전/landmark25_all_cohorts.csv` 하나로 저장한다.
`cohort_status_25`로 모델 대상과 세 가지 제외 세그먼트를 구분한다. 제외 세그먼트는
모델 피처 산출 대상이 아니므로 VLE·평가 피처와 타깃은 결측으로 유지한다.

In [9]:
integrated_table_25['cohort_status_25'] = 'model_eligible'

excluded_parts = []
for status, excluded in [
    ('excluded_early_leaver', excluded_early_leaver_25),
    ('excluded_ambiguous_withdrawn', excluded_ambiguous_withdrawn_25),
    ('excluded_late_registration', excluded_late_registration_25),
]:
    part = excluded.copy()
    part['landmark_day'] = LANDMARK_N
    part['eligible_at_25'] = 0
    part['target_churn_after_25'] = np.nan
    part['cohort_status_25'] = status
    excluded_parts.append(part)

all_cohorts_25 = pd.concat(
    [integrated_table_25, *excluded_parts], ignore_index=True, sort=False
)
output_cols = final_cols + ['cohort_status_25']
all_cohorts_25 = all_cohorts_25.reindex(columns=output_cols)

assert len(all_cohorts_25) == 32_593, '전체 코호트 행 수 불일치'
assert all_cohorts_25.duplicated(KEYS).sum() == 0, '전체 코호트 키 중복 발생'
assert (all_cohorts_25['eligible_at_25'] == 1).sum() == 27_661, '모델 대상 행 수 불일치'
assert all_cohorts_25.loc[all_cohorts_25['eligible_at_25'] == 1, 'target_churn_after_25'].sum() == 5_247, '양성 건수 불일치'

INTEGRATED_DIR.mkdir(parents=True, exist_ok=True)
all_cohorts_25.to_csv(INTEGRATED_DIR / 'landmark25_all_cohorts.csv', index=False)

print('저장 완료:', INTEGRATED_DIR / 'landmark25_all_cohorts.csv')
print(all_cohorts_25['cohort_status_25'].value_counts())

저장 완료: ..\..\CSV_files\통합 버전\landmark25_all_cohorts.csv
cohort_status_25
model_eligible                  27661
excluded_early_leaver            4819
excluded_ambiguous_withdrawn       93
excluded_late_registration         20
Name: count, dtype: int64


## 9. 다음 단계

```
01_Window_Definition → [02_Integrated_Table] → 03_Landmark25_EDA → 04_Missing_Outlier → 05_Collinearity_Recheck → 03_Modeling
```

다음 노트북은 `03_Landmark25_EDA.ipynb`다. 이 노트북이 저장한 `landmark25_all_cohorts.csv`를 **전처리 결정 전에** 과목·학기 단위로 탐색한다.
여기서 쓰는 평가 기회·이월 정의(5절)가 실제 과목 일정과 맞는지도 그 노트북 5·6절에서 검증한다.